In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from src.agent import HelpDeskAgent
from src.configs import Settings
from src.tools.search_xyz_manual import search_xyz_manual
from src.tools.search_xyz_qa import search_xyz_qa

settings = Settings()

In [3]:
agent = HelpDeskAgent(
    settings=settings,
    tools=[search_xyz_manual, search_xyz_qa]
)

In [4]:
question = """
お世話になっております。

現在、XYZシステムの利用を検討しており、以下の2点についてご教示いただければと存じます。

1. パスワードに利用可能な文字の制限について
当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

2. 最新リリースの取得方法について
最新のアップデート情報をどのように確認・取得できるかについてもお教えいただけますと幸いです。

お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。
"""

In [5]:
input_data = {"question": question}

plan_result = agent.create_plan(state=input_data)

2026-04-29 00:32:34,050 INFO Starting plan generation process...
2026-04-29 00:32:34,054 INFO Sending request to OpenAI...
2026-04-29 00:32:38,722 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-29 00:32:38,805 INFO Successfully received response from OpenAI
2026-04-29 00:32:38,807 INFO Plan generation complete


In [6]:
plan_result["plan"]

['XYZシステムにおけるパスワード設定時の利用可能な文字の種類と範囲について調べる：英数字、記号、最大・最小文字数などの仕様を確認する',
 'XYZシステムの最新リリース情報およびアップデート取得方法について調べる：公式サイトや通知方法、ダウンロード手順などを確認する']

In [7]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "is_completed": False
}

In [8]:
select_tool_result = agent.select_tools(state=input_data)

2026-04-29 00:32:42,450 INFO Starting tool selection process...
2026-04-29 00:32:42,896 INFO Sending request to OpenAI...
2026-04-29 00:32:43,771 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-29 00:32:43,782 INFO Successfully received response from OpenAI
2026-04-29 00:32:43,788 INFO Tool selecting complete!


In [9]:
select_tool_result

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [10]:
select_tool_result["messages"][-1]

{'role': 'assistant',
 'tool_calls': [{'id': 'call_IDjfFMxozubE3J90bZQ8Yu4M',
   'function': {'arguments': '{"keywords":"パスワード 使用可能な文字"}',
    'name': 'search_xyz_manual'},
   'type': 'function'}]}

In [11]:
select_tool_result["messages"]

[{'role': 'system',
  'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
 {'role': 'user',
  'content': "\nユーザーの元の質問: \nお世話になっております。\n\n現在、XYZシステムの利用

In [12]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": select_tool_result["messages"],
    "is_completed": False
}

In [13]:
tool_results = agent.execute_tools(state=input_data)

2026-04-29 00:35:01,904 INFO Starting tool execution process...
2026-04-29 00:35:01,918 INFO Searching XYZ manual by keyword: {"keywords":"パスワード 使用可能な文字"}
2026-04-29 00:35:01,994 INFO POST http://localhost:9200/documents/_search [status:200 duration:0.070s]
2026-04-29 00:35:01,997 INFO Search results: 10 hits
2026-04-29 00:35:02,001 INFO Finished searching XYZ manual by keyword
2026-04-29 00:35:02,004 INFO Tool execution complete!


In [14]:
tool_results["tool_results"][0][0].results

[SearchOutput(file_name='XYZシステム統合ユーザーマニュアル.pdf', content='最低8⽂字以上の⻑さ\n⼤⽂字と⼩⽂字の両⽅を含む\n少なくとも1 つの数字を含む\n少なくとも1 つの特殊⽂字（ !@#$%^&* 等）を含む\nシステムは、これらの基準に基づい てパスワードの強度を「弱」 「中」「強」\nの3段階で評価します 。「弱」と判定されたパス ワードは受け付けられ ず、ユ\nーザーは強度の⾼いパス ワードを設定するよう促されます 。また、過去に 使'),
 SearchOutput(file_name='XYZシステムリリースノート.pdf', content='の変更や、必要に応じて⾃⾝のアカウントを削除することが可能となりました。従\n来のシステムでは、ユーザーからパスワード管理機能に関する要望が多く、⾃⼰管\n理ツールが求められていました。\nまた、プロジェクトテンプレートを利⽤することで、新規プロジェクトの⽴ち上げ\nが迅速に⾏えるようになりました。従来のプロジェクト管理では、多くの設定を最'),
 SearchOutput(file_name='XYZシステム統合ユーザーマニュアル.pdf', content='ます。\nオンラインヘル プセンター：24時間365 ⽇利⽤可能なヘル プセンター では、シ ス\nテムの使⽤⽅法や⼀般的な問題に関す る問い合わせを受け付けています 。\n定期的なウェビ ナー：新機能の紹介や⾼度な使⽤テクニックを学ぶためのオン\nラインセミナーを開催し ています 。\n2. システムへのアクセスとログイン')]

In [15]:
tool_results

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [16]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": select_tool_result["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False
}

In [17]:
subtask_answer = agent.create_subtask_answer(state=input_data)

2026-04-29 00:37:18,543 INFO Starting subtask answer creation process...
2026-04-29 00:37:18,548 INFO Sending request to OpenAI...
2026-04-29 00:37:22,366 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-29 00:37:22,373 INFO Successfully received response from OpenAI
2026-04-29 00:37:22,375 INFO Subtask answer creation complete!


In [18]:
subtask_answer

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [19]:
print(subtask_answer["subtask_answer"])

2. サブタスク回答

XYZシステムのパスワード設定に関する仕様は以下の通りです。

- パスワードは最低8文字以上であることが求められています。
- 英字の大文字と小文字の両方を含む必要があります。
- 少なくとも1つの数字を含める必要があります。
- 少なくとも1つの特殊文字（例: !@#$%^&*）を含む必要があります。

このパスワード規定に基づいて、システムはパスワードの強度を「弱」「中」「強」の3段階で評価し、「弱」と判定されたパスワードは受け付けられません。ユーザーは強度の高いパスワードを設定するよう促されます。


In [20]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": select_tool_result["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
    "subtask_answer": subtask_answer["subtask_answer"]
}

In [21]:
reflection_result = agent.reflect_subtask(state=input_data)

2026-04-29 00:38:48,652 INFO Starting refrection process...
2026-04-29 00:38:48,657 INFO Sending request to OpenAI...
2026-04-29 00:38:49,981 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-29 00:38:49,989 INFO Successfully received response from OpenAI
2026-04-29 00:38:49,991 INFO Reflection complete!


In [22]:
reflection_result

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [23]:
print(reflection_result["messages"][2]["tool_calls"][0]["function"]["name"])

search_xyz_manual


In [24]:
print("is_completed =", reflection_result["reflection_results"][0].is_compiled)
print("advice =", reflection_result["reflection_results"][0].advice)

is_completed = True
advice = 
